# Soft Robot Evolution — Technical Deep Dive

This notebook explains every component of the soft robot evolution system in `c:/SoftRobotEvolution`:
the genome encoding, the physics model, the fitness function, the genetic operators,
the Age-Fitness Pareto selection algorithm, and the parallel evaluation architecture.

All plots use synthetic / illustrative data and run standalone (no evolution run needed).

---
**Key files:**
- `run_tendon_evolution.py` — main entry point, evolution loop, mutation, crossover, Pareto selection
- `src/evolution/genome_config.py` — voxel grid constants, growth algorithm, LCC connectivity
- `src/evolution/controllers.py` — CPGController (one oscillator per active tendon)
- `src/physics/mujoco_tendon_physics.py` — MuJoCo simulation, fitness computation
- `src/physics/mujoco_tendon_converter.py` — voxel grid → MuJoCo XML with springs
- `src/evolution/mujoco_tendon_evaluator.py` — multiprocessing pool wrapper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
from collections import deque
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Canonical material colours (match the physics engine)
MAT_COLOR = {0: 'white', 1: '#22cc44', 2: '#cc2222', 3: '#22cccc', 4: '#2244cc'}
MAT_LABEL = {
    1: 'mat1 — Active  0° (green)',
    2: 'mat2 — Active 180° (red)',
    3: 'mat3 — Soft passive (cyan)',
    4: 'mat4 — Stiff passive (blue)',
}
print('Setup complete.')

---
## 1. System Overview

The system evolves soft robots represented as **3-D voxel grids**. Each robot is evaluated
in a MuJoCo physics simulation; the fitness is how far it crawls. Selection uses
**Age-Fitness Pareto** to balance exploitation (good movers) vs exploration (young diversity).

```
┌─────────────────────────────────────────────────────────────────┐
│                     EVOLUTION LOOP  (run_tendon_evolution.py)   │
│                                                                  │
│  ┌──────────┐   mutate/   ┌──────────┐  MuJoCo   ┌──────────┐  │
│  │Population│─crossover──▶│ Children │──Physics──▶│Fitnesses │  │
│  │ N robots │             │  N-new   │  (30 CPU   │          │  │
│  │ (genome +│             │ + inject │   workers) │ cached   │  │
│  │ controller│◀──Pareto───│  12 rand │            │ for surv.│  │
│  │ + age)   │  truncate   └──────────┘            └──────────┘  │
│  │  top N   │  2N → N                                            │
│  └──────────┘                                                    │
│       │ every gen: save best_robot_tendon.pkl (Ctrl+C safe)      │
└───────┼─────────────────────────────────────────────────────────┘
        ▼
  visualize_tendon_best.py   →   MuJoCo viewer
```

---
## 2. Genome Encoding

Each robot is a **16 × 16 × 16 numpy int8 array**. Every cell is one voxel:

| Value | Meaning | Spring kp | Actuation |
|---|---|---|---|
| 0 | Empty | — | — |
| 1 | Active 0°   (green)  | 100 N/m | sin(ωt + 0) |
| 2 | Active 180° (red)    | 100 N/m | sin(ωt + π) |
| 3 | Soft passive (cyan)  |  50 N/m | none (holds shape) |
| 4 | Stiff passive (blue) | 100 N/m | none (holds shape) |

Only **face-adjacent** (Manhattan distance = 1) voxel pairs are connected by springs.
Diagonal connections are excluded — they generated 4× too much force and launched robots.

**Usable space:** Interior indices 1–14 → 14³ = 2744 possible positions.
Each robot contains 20–100 voxels (enforced by the growth algorithm and mutation).

In [ ]:
# --- Simulate the growth algorithm to create a synthetic robot ---
np.random.seed(42)
SHAPE = (16, 16, 16)
FACE  = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]

def grow_robot(n_voxels=50, seed=42):
    rng  = np.random.default_rng(seed)
    grid = np.zeros(SHAPE, dtype=np.int8)
    sx, sy, sz = rng.integers(3, 13, size=3)
    grid[sx, sy, sz] = rng.integers(1, 5)
    frontier = [(sx, sy, sz)]
    placed = 1
    while placed < n_voxels and frontier:
        fi = rng.integers(len(frontier))
        fx, fy, fz = frontier[fi]
        cands = []
        for dx, dy, dz in FACE:
            nx, ny, nz = fx+dx, fy+dy, fz+dz
            if 1<=nx<15 and 1<=ny<15 and 1<=nz<15 and grid[nx,ny,nz]==0:
                cands.append((nx,ny,nz))
        if cands:
            cx,cy,cz = cands[rng.integers(len(cands))]
            grid[cx,cy,cz] = rng.integers(1,5)
            frontier.append((cx,cy,cz))
            placed += 1
        else:
            frontier.pop(fi)
    return grid

robot = grow_robot(50)

fig = plt.figure(figsize=(14, 5))

# --- 3D scatter ---
ax3d = fig.add_subplot(1, 2, 1, projection='3d')
for mat in [1, 2, 3, 4]:
    pts = np.argwhere(robot == mat)
    if len(pts):
        ax3d.scatter(pts[:,0], pts[:,1], pts[:,2],
                     c=MAT_COLOR[mat], s=60, alpha=0.8,
                     label=MAT_LABEL[mat], edgecolors='k', linewidths=0.3)
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z (height)')
ax3d.set_title(f'Robot genome — {(robot!=0).sum()} voxels in 16x16x16 grid')
ax3d.legend(loc='upper left', fontsize=8)

# --- XZ cross-section (side view) ---
ax2d = fig.add_subplot(1, 2, 2)
mid_y = robot.shape[1] // 2
# find a y-slice that has voxels
for dy in range(5):
    if robot[:, mid_y+dy, :].any():
        mid_y = mid_y+dy; break
    elif robot[:, mid_y-dy, :].any():
        mid_y = mid_y-dy; break
slice2d = robot[:, mid_y, :]
cmap_list = [MAT_COLOR[i] for i in range(5)]
cmap5 = mcolors.ListedColormap(cmap_list)
ax2d.imshow(slice2d.T, origin='lower', cmap=cmap5, vmin=0, vmax=4, aspect='equal')
ax2d.set_title(f'XZ cross-section at y={mid_y} (side view)')
ax2d.set_xlabel('X'); ax2d.set_ylabel('Z (height)')
patches = [mpatches.Patch(color=MAT_COLOR[i], label=MAT_LABEL[i]) for i in [1,2,3,4]]
patches.insert(0, mpatches.Patch(color='white', label='0 — Empty', ec='grey'))
ax2d.legend(handles=patches, loc='upper right', fontsize=8)
ax2d.grid(False)

plt.tight_layout()
plt.show()
mats, cnts = np.unique(robot[robot!=0], return_counts=True)
print('Material breakdown:', dict(zip(mats, cnts)))

---
## 3. Initialisation — Growth Algorithm

Robots are initialised with a **seeded face-growth algorithm** that guarantees connectivity:

1. Place one voxel at a random interior position (seed)
2. Maintain a *frontier* of occupied voxels that still have empty face-neighbours
3. Pick a random frontier voxel; add one random empty face-neighbour with a random material
4. Repeat until target voxel count is reached

This guarantees the body is **face-connected** (no floating pieces), matching the spring topology.
After any mutation or crossover, `keep_largest_component()` (BFS 6-connectivity) is applied
to drop any orphaned clusters.

In [ ]:
# Visualise growth at 4 stages
snapshots = [1, 5, 20, 50]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

np.random.seed(7)
FACE = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]

full_grid = np.zeros((16,16,16), dtype=np.int8)
sx, sy, sz = 8, 8, 8
full_grid[sx,sy,sz] = np.random.randint(1,5)
frontier_grow = [(sx,sy,sz)]
placed = 1
snap_grids = {}
snap_grids[1] = full_grid.copy()

while placed < 50:
    fi = np.random.randint(len(frontier_grow))
    fx,fy,fz = frontier_grow[fi]
    cands = []
    for dx,dy,dz in FACE:
        nx,ny,nz = fx+dx,fy+dy,fz+dz
        if 1<=nx<15 and 1<=ny<15 and 1<=nz<15 and full_grid[nx,ny,nz]==0:
            cands.append((nx,ny,nz))
    if cands:
        cx,cy,cz = cands[np.random.randint(len(cands))]
        full_grid[cx,cy,cz] = np.random.randint(1,5)
        frontier_grow.append((cx,cy,cz))
        placed += 1
        if placed in snapshots:
            snap_grids[placed] = full_grid.copy()
    else:
        frontier_grow.pop(fi)

cmap5 = mcolors.ListedColormap([MAT_COLOR[i] for i in range(5)])
for ax, n in zip(axes, snapshots):
    g = snap_grids.get(n, full_grid)
    mid = g.shape[1]//2
    ax.imshow(g[:,mid,:].T, origin='lower', cmap=cmap5, vmin=0, vmax=4, aspect='equal')
    ax.set_title(f'Step {n}: {(g!=0).sum()} voxels')
    ax.set_xlabel('X'); ax.set_ylabel('Z')
    ax.grid(False)

plt.suptitle('Growth algorithm — XZ cross-section at y=8', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Physics Engine (MuJoCo)

Each voxel becomes a **free rigid body** in MuJoCo. Adjacent voxels (face-only, 6-connectivity)
are connected by **spatial tendons + position actuators**, forming a spring network.

### Key parameters

| Parameter | Value | Reason |
|---|---|---|
| Timestep dt | 0.0005 s | Stability with stiff springs |
| Voxel size | 0.01 m (1 cm) | Physical scale |
| Voxel density | 1000 kg/m³ | Water density — standard for soft robot sims |
| Voxel mass | 1.0 g | 1000 × (0.01)³ |
| kp active/stiff | 100 N/m | Spring stiffness |
| kp soft passive | 50 N/m | 2× softer |
| kv damping | 0.1 N·s/m | Prevents resonance; stability bound: kv < m/(6·dt) = 0.33 |
| Actuation freq | 10 Hz | ω = 2π × 10 rad/s |
| Actuation amp | ±8% | Ctrl = rest_length × (1 ± 0.08 × signal) |
| Gravity | 9.81 m/s² | Standard |
| Ground friction | 0.6 | Sliding friction coefficient |

### Why face-only springs?

The original 26-neighbour (full 3D adjacency) model generated forces of 200+g per voxel —
robots launched themselves into the air immediately. Face-only (6 springs per interior voxel)
reduces peak force by ~4× and matches the Cheney 2013 / Voxelyze standard.

In [ ]:
# Force budget analysis: spring force vs gravity
kp = 100.0          # N/m
amp = 0.08          # 8%
rest_len = 0.01     # m
density = 1000.0    # kg/m^3
voxel_size = 0.01   # m
mass = density * voxel_size**3  # 0.001 kg
g = 9.81

n_neighbors = np.arange(1, 7)
force_per_spring = kp * amp * rest_len   # 0.08 N per spring
total_force = n_neighbors * force_per_spring
accel_g = total_force / (mass * g)  # in units of g

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Left: acceleration vs number of neighbours
ax1.bar(n_neighbors, accel_g, color='steelblue', alpha=0.8, edgecolor='k')
ax1.axhline(1.0, color='red', ls='--', lw=1.5, label='1 g (gravity)')
ax1.axhline(10.0, color='orange', ls='--', lw=1.5, label='10 g')
ax1.set_xlabel('Number of active face-neighbours')
ax1.set_ylabel('Peak acceleration (units of g)')
ax1.set_title('Spring force budget per voxel')
ax1.legend()
ax1.set_xticks(n_neighbors)

# Right: compare old density vs new density
old_mass = 200 * voxel_size**3
labels = ['Old\n200 kg/m³', 'Current\n1000 kg/m³']
masses = [old_mass, mass]
accs_6 = [6 * force_per_spring / (m * g) for m in masses]
bars = ax2.bar(labels, accs_6, color=['#cc4444', '#44aa44'], alpha=0.8, edgecolor='k')
ax2.axhline(1.0, color='red', ls='--', lw=1.5, label='1 g')
ax2.set_ylabel('Peak acceleration (g) — worst case 6 neighbours')
ax2.set_title('Effect of density on force budget')
ax2.legend()
for bar, val in zip(bars, accs_6):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.5, f'{val:.0f}g',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Voxel mass: {mass*1000:.1f} g   |  Force per spring: {force_per_spring:.4f} N')
print(f'6 neighbours: {accel_g[-1]:.1f} g acceleration (current)  vs  {6*force_per_spring/(old_mass*g):.0f}g (old)')

---
## 5. Actuation — How Robots Move

Active voxels (mat1, mat2) generate **sinusoidal length changes** in their connected springs:

```
ctrl(t) = rest_length × (1 + amplitude × signal(t))

Without CPG controller (default sinusoidal):
  signal(t) = sin(ω·t + base_phase)
  mat1 base_phase = 0      →  sin(ω·t)
  mat2 base_phase = π      →  sin(ω·t + π)  =  −sin(ω·t)

With CPG controller:
  signal_k(t) = A_k · sin(2π·f_k·t + φ_k) + coupling terms
```

mat1 and mat2 are **180° anti-phase**: when mat1 tendons contract, mat2 tendons extend.
This creates a **peristaltic / wave-like motion** if the materials are arranged alternately.

In [ ]:
t = np.linspace(0, 0.4, 1000)   # 0.4 s = 4 actuation cycles at 10 Hz
omega = 2 * np.pi * 10.0         # 10 Hz
amp   = 0.08
rest  = 0.01                     # 1 cm rest length

ctrl_mat1 = rest * (1 + amp * np.sin(omega * t))
ctrl_mat2 = rest * (1 + amp * np.sin(omega * t + np.pi))

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

# Top: control signals
axes[0].plot(t*1000, ctrl_mat1*100, color=MAT_COLOR[1], lw=2, label='mat1 (0°)')
axes[0].plot(t*1000, ctrl_mat2*100, color=MAT_COLOR[2], lw=2, ls='--', label='mat2 (180°)')
axes[0].axhline(rest*100, color='grey', lw=1, ls=':', label='rest length')
axes[0].fill_between(t*1000, rest*100, ctrl_mat1*100, alpha=0.15, color=MAT_COLOR[1])
axes[0].fill_between(t*1000, rest*100, ctrl_mat2*100, alpha=0.15, color=MAT_COLOR[2])
axes[0].set_ylabel('Target tendon length (cm)')
axes[0].set_title('Actuation signals — 10 Hz, ±8% of rest length')
axes[0].legend()
axes[0].set_ylim(0.88, 1.12)

# Bottom: if mat1-mat2 pair, the relative extension
relative = ctrl_mat1 - ctrl_mat2
axes[1].plot(t*1000, relative*100, color='purple', lw=2, label='mat1 − mat2 (relative stretch)')
axes[1].axhline(0, color='grey', lw=1, ls=':')
axes[1].fill_between(t*1000, 0, relative*100, alpha=0.2, color='purple')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Relative length difference (cm)')
axes[1].set_title('Anti-phase pair: maximum pushing/pulling force when mat1 & mat2 are adjacent')
axes[1].legend()

# Mark one period
period_ms = 100  # 10 Hz
for ax in axes:
    ax.axvspan(0, period_ms, alpha=0.05, color='gold', label='1 cycle (100 ms)')

plt.tight_layout()
plt.show()

---
## 6. CPG Controller

The **CPGController** (`src/evolution/controllers.py`) replaces the hard-coded sinusoidal
with one **evolvable oscillator per active tendon**:

```python
base_signal_k(t) = A_k * sin(2π * f_k * t + φ_k)
output_k(t)      = base_k + 0.1 * Σ_j W_kj * output_j(t-dt)   # weak coupling
ctrl_k(t)        = clip(output_k, -1, 1)
```

**Evolvable parameters** (all mutated by Gaussian perturbation):
- `frequencies` — per-tendon frequency, clipped to [0.1, 5.0] Hz
- `amplitudes`  — per-tendon amplitude, clipped to [0.0, 1.0]
- `phases`      — per-tendon phase offset, modulo 2π
- `coupling`    — N×N weight matrix (diagonal=0), clipped to [-0.5, 0.5]

**Controller crossover**: child phases and frequencies = average of both parents (if sizes match).
If sizes differ (different body topology), a fresh controller is created.

In [ ]:
# Simulate 4 CPG oscillators with different f and phase
np.random.seed(5)
n_osc   = 4
freqs   = np.array([1.2, 2.1, 1.8, 0.9])
amps    = np.array([0.9, 0.7, 0.8, 1.0])
phases  = np.array([0.0, np.pi/3, np.pi, 3*np.pi/2])
coupling = np.random.randn(n_osc, n_osc) * 0.1
np.fill_diagonal(coupling, 0)

dt = 0.001
T  = 3.0
times = np.arange(0, T, dt)
states = np.zeros(n_osc)
signals = np.zeros((len(times), n_osc))

for i, tt in enumerate(times):
    base = amps * np.sin(2*np.pi*freqs*tt + phases)
    states = base + 0.1 * coupling @ states
    signals[i] = np.clip(states, -1, 1)

fig, axes = plt.subplots(n_osc, 1, figsize=(13, 7), sharex=True)
osc_labels = ['Tendon #1 (body front)', 'Tendon #2 (body mid-L)',
               'Tendon #3 (body mid-R)', 'Tendon #4 (body rear)']
colors = ['#22cc44','#cc2222','#22cccc','#2244cc']

for k in range(n_osc):
    axes[k].plot(times, signals[:,k], color=colors[k], lw=1.5)
    axes[k].axhline(0, color='grey', lw=0.8, ls=':')
    axes[k].set_ylabel(f'Signal', fontsize=9)
    axes[k].set_ylim(-1.2, 1.2)
    axes[k].set_yticks([-1, 0, 1])
    axes[k].text(0.01, 0.85, f'{osc_labels[k]}  |  f={freqs[k]:.1f}Hz  φ={phases[k]:.2f}rad',
                 transform=axes[k].transAxes, fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor=colors[k], alpha=0.2))

axes[-1].set_xlabel('Time (s)')
fig.suptitle('CPG Controller — 4 oscillators driving 4 active tendons', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Fitness Function

```python
# In mujoco_tendon_physics.py  →  get_fitness()

# 1. Settle phase: run 0.5 s of actuation (let robot find its footing)
# 2. Record start position and ground threshold:
initial_pos = get_COM_position()
ground_threshold = max(initial_pos.z * 3.0, 0.05)  # 5 cm minimum

# 3. Run simulation_time seconds; count grounded steps
grounded_steps = 0
for each step:
    if COM.z < ground_threshold:
        grounded_steps += 1

# 4. Compute fitness
ground_fraction = grounded_steps / total_steps      # 0.0 – 1.0
horizontal_dist = |final_pos_xy − initial_pos_xy|   # metres
fitness = horizontal_dist * ground_fraction
```

**Why ground_fraction?**  Without it, the EA discovers that launching the robot into the air
gives huge horizontal displacement at zero energy cost. `ground_fraction` penalises airborne motion.
A robot that crawls continuously scores full credit; one that jumps 2 m into the air scores ~0.

In [ ]:
np.random.seed(3)
dt_sim = 0.01
T_sim  = 5.0
t_sim  = np.arange(0, T_sim, dt_sim)
n_steps = len(t_sim)

# --- Scenario A: crawling robot (stays on ground) ---
com_z_crawl = 0.008 + 0.003*np.sin(2*np.pi*10*t_sim) + np.random.normal(0,0.001,n_steps)
com_z_crawl = np.clip(com_z_crawl, 0.003, None)
com_x_crawl = 0.12 * t_sim / T_sim   # moves 12 cm

# --- Scenario B: jumping robot (launches at t=1s) ---
com_z_jump = 0.008 + 0.003*np.sin(2*np.pi*10*t_sim)
launch_idx = int(1.0/dt_sim)
for i in range(launch_idx, n_steps):
    elapsed = (i-launch_idx)*dt_sim
    com_z_jump[i] = max(0.005, 0.008 + 3.0*elapsed - 4.9*elapsed**2)
com_x_jump = np.zeros(n_steps)
com_x_jump[:launch_idx] = 0.01 * t_sim[:launch_idx]
com_x_jump[launch_idx:] = com_x_jump[launch_idx-1] + 0.8*(t_sim[launch_idx:]-1.0)

threshold = max(com_z_crawl[:int(0.5/dt_sim)].mean() * 3.0, 0.05)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

for col, (com_z, com_x, label) in enumerate([
        (com_z_crawl, com_x_crawl, 'Crawling robot'),
        (com_z_jump,  com_x_jump,  'Jumping / flying robot')]):

    grounded = com_z < threshold
    gf = grounded.mean()
    hdist = abs(com_x[-1] - com_x[0])
    fitness = hdist * gf

    # COM height
    ax = axes[0, col]
    ax.plot(t_sim, com_z*100, lw=1.5, color='steelblue')
    ax.axhline(threshold*100, color='red', ls='--', lw=1.5, label=f'threshold={threshold*100:.1f} cm')
    ax.fill_between(t_sim, 0, com_z*100, where=grounded,
                    alpha=0.3, color='green', label=f'grounded ({gf*100:.0f}%)')
    ax.fill_between(t_sim, 0, com_z*100, where=~grounded,
                    alpha=0.3, color='red', label=f'airborne ({(1-gf)*100:.0f}%)')
    ax.set_title(f'{label}  →  fitness = {fitness:.3f} m')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('COM height (cm)')
    ax.legend(fontsize=8)

    # X displacement
    ax2 = axes[1, col]
    ax2.plot(t_sim, com_x*100, lw=1.5, color='darkorange')
    ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Horizontal position (cm)')
    ax2.set_title(f'X displacement:  raw={hdist*100:.1f}cm  ×  gf={gf:.2f}  =  {fitness*100:.1f}cm fitness')

plt.tight_layout()
plt.show()

---
## 8. Genetic Operators

### 8a. Mutation

```python
# run_tendon_evolution.py  →  mutate(grid, rate=0.3)
if random() < rate:               # 30% chance to apply any mutation
    for _ in range(randint(1,4)): # 1–4 individual operations
        if random() < 0.5:
            # ADD or CHANGE a voxel at a random interior position
            grid[x,y,z] = random_material()
        else:
            # REMOVE a random occupied voxel (if body has > 3 voxels)
            grid[x,y,z] = 0
grid = keep_largest_component(grid)   # drop orphaned clusters (BFS)
if count(grid) < 3: return original  # safety guard
```

**Mutation rate = 0.3** means 70% of children are identical to their parent (only the controller mutates).

### 8b. Crossover

```python
# Planar split crossover  →  crossover_3d(p1, p2)
axis  = random choice of X, Y, or Z
split = random integer in [1, 14]  (interior range)
child[x,y,z] = p1[x,y,z]  if coord[axis] < split
               p2[x,y,z]  otherwise
child = keep_largest_component(child)
```

The child gets the "front half" of parent 1 and "back half" of parent 2 (or left/right, up/down).

In [ ]:
# Visualise crossover on two parent bodies
np.random.seed(11)
p1 = grow_robot(40, seed=11)
p2 = grow_robot(40, seed=22)

axis  = 0       # split along X axis
split = 8       # split at x=8 (midpoint)
child = np.where(np.arange(16)[:,None,None] < split, p1, p2).astype(np.int8)

# For LCC, keep largest connected component
def keep_lcc(grid):
    occupied = set(map(tuple, np.argwhere(grid!=0).tolist()))
    if not occupied: return grid.copy()
    unvisited = set(occupied); comps = []
    FACE6 = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]
    while unvisited:
        start = next(iter(unvisited))
        comp = []; q = deque([start]); unvisited.discard(start)
        while q:
            x,y,z = q.popleft(); comp.append((x,y,z))
            for dx,dy,dz in FACE6:
                nb=(x+dx,y+dy,z+dz)
                if nb in unvisited: unvisited.discard(nb); q.append(nb)
        comps.append(comp)
    largest = max(comps, key=len)
    result = np.zeros_like(grid)
    for x,y,z in largest: result[x,y,z]=grid[x,y,z]
    return result

child = keep_lcc(child)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cmap5 = mcolors.ListedColormap([MAT_COLOR[i] for i in range(5)])
mid = 8
titles = ['Parent 1', 'Parent 2', f'Child (split at X={split})']
grids  = [p1, p2, child]

for ax, g, title in zip(axes, grids, titles):
    ax.imshow(g[:,mid,:].T, origin='lower', cmap=cmap5, vmin=0, vmax=4, aspect='equal')
    ax.set_title(f'{title}  ({(g!=0).sum()} voxels)')
    ax.set_xlabel('X'); ax.set_ylabel('Z')
    ax.grid(False)
    if 'Child' in title:
        ax.axvline(split-0.5, color='white', lw=2, ls='--', alpha=0.9, label=f'split X={split}')
        ax.legend(fontsize=9)

plt.suptitle('Planar-split crossover (axis=X, split=8) — XZ cross-section at y=8')
plt.tight_layout()
plt.show()

---
## 9. Age-Fitness Pareto Selection

**Reference:** Schmidt & Lipson, *"Age-Fitness Pareto Optimization"*, GECCO 2010
(paper in `Documentation/Age-Fitness Pareto Optimization_article.pdf`)

### The problem with pure fitness selection
Standard tournament selection drives the whole population toward one local optimum within
10–20 generations. Once a "good enough" robot dominates, diversity collapses and exploration stops.

### The Age-Fitness idea
Maintain a **second objective: minimise age**. Each robot carries an `age` counter:

| Event | Age change |
|---|---|
| New random injection | age = 1 |
| Bred child | age = max(parent1.age, parent2.age) |
| Surviving one generation | age += 1 |

**Dominance rule:** Individual A dominates B iff:
- `A.fitness >= B.fitness` AND `A.age <= B.age` (A is at least as good on both)
- At least one is strict (A is strictly better on one dimension)

**Effect:** Old high-fitness individuals lose their Pareto dominance to younger mid-fitness ones.
Fresh random robots (age=1) are never dominated by age alone, so they always survive at least
one generation — continuously injecting genetic diversity.

### Each generation
1. Breed `N − n_inject` children from the current population (tournament selection for parents)
2. Inject `n_inject=12` completely fresh random robots (age=1)
3. Pool = current N (cached fitness) + new N (newly evaluated) = 2N total
4. **Pareto-sort** on (fitness↑, age↓); keep top N by front rank, break ties by fitness
5. `ages += 1` for all survivors

In [ ]:
def pareto_front_labels(fitnesses, ages):
    """Return Pareto front index (0=best) for each individual."""
    n = len(fitnesses)
    dom_count = np.zeros(n, dtype=int)
    dom_over  = [[] for _ in range(n)]
    for i in range(n):
        for j in range(i+1, n):
            fi,ai = fitnesses[i], ages[i]
            fj,aj = fitnesses[j], ages[j]
            if fi>=fj and ai<=aj and (fi>fj or ai<aj):
                dom_count[j]+=1; dom_over[i].append(j)
            elif fj>=fi and aj<=ai and (fj>fi or aj<ai):
                dom_count[i]+=1; dom_over[j].append(i)
    front = np.full(n, -1, dtype=int)
    current = [i for i in range(n) if dom_count[i]==0]
    f_idx = 0
    remaining = dom_count.copy()
    while current:
        for i in current: front[i] = f_idx
        nxt = []
        for i in current:
            for j in dom_over[i]:
                remaining[j] -= 1
                if remaining[j] == 0: nxt.append(j)
        current = nxt; f_idx += 1
    return front

# Simulate a population at generation 20
np.random.seed(42)
N = 60
# Older individuals tend to be fitter (selection pressure), younger more diverse
ages_pop  = np.random.randint(1, 25, N)
fits_pop  = np.clip(np.log(ages_pop+1)*0.3 + np.random.normal(0, 0.15, N), 0, None)
# Inject some fresh random low-fitness individuals
fits_pop[-8:] = np.random.uniform(0, 0.1, 8)
ages_pop[-8:] = 1

front_labels = pareto_front_labels(fits_pop, ages_pop)
n_fronts = front_labels.max()+1
front_colors = plt.cm.RdYlGn_r(np.linspace(0, 0.85, min(n_fronts, 6)))

fig, (ax_main, ax_explain) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Pareto scatter
for f in range(min(n_fronts, 6)):
    mask = front_labels == f
    label = f'Front {f+1} ({mask.sum()} individuals)' if f<4 else f'Front {f+1}+'
    ax_main.scatter(ages_pop[mask], fits_pop[mask],
                    c=[front_colors[min(f,5)]], s=80, alpha=0.8,
                    edgecolors='k', linewidths=0.5, label=label, zorder=3)

# Mark injected robots
inj_mask = ages_pop == 1
ax_main.scatter(ages_pop[inj_mask], fits_pop[inj_mask],
                s=120, facecolors='none', edgecolors='gold', linewidths=2,
                label='Injected (age=1)', zorder=4)

ax_main.set_xlabel('Age (generations of genetic material)')
ax_main.set_ylabel('Fitness (m crawled)')
ax_main.set_title(f'Population at gen 20  —  {n_fronts} Pareto fronts\n(Front 1 = non-dominated, kept first)')
ax_main.legend(fontsize=8, loc='upper left')

# Right: dominance illustration
ex_fits = np.array([0.8, 0.5, 0.8, 0.3])
ex_ages = np.array([5,   5,   10,  2  ])
ex_labels = ['A', 'B', 'C', 'D']
ex_colors = ['#22cc44','#cc2222','#cc8800','#2244cc']

for i, (f,a,lbl,c) in enumerate(zip(ex_fits, ex_ages, ex_labels, ex_colors)):
    ax_explain.scatter(a, f, s=200, c=c, zorder=5, edgecolors='k', lw=1.5)
    ax_explain.annotate(lbl, (a,f), xytext=(a+0.3, f+0.01), fontsize=13, fontweight='bold', color=c)

# Dominance arrows
# A dominates B? A.f=0.8 >= B.f=0.5 and A.age=5 <= B.age=5: yes (strict in fitness)
ax_explain.annotate('', xy=(ex_ages[1],ex_fits[1]), xytext=(ex_ages[0],ex_fits[0]),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax_explain.text(4.0, 0.63, 'A dominates B\n(same age, higher fitness)', fontsize=9)

# A dominates C? A.f=0.8 >= C.f=0.8 and A.age=5 <= C.age=10: yes (strict in age)
ax_explain.annotate('', xy=(ex_ages[2],ex_fits[2]), xytext=(ex_ages[0],ex_fits[0]),
                    arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.5))
ax_explain.text(6.0, 0.82, 'A dominates C\n(same fitness, younger)', fontsize=9)

# D: nobody dominates D (low age=2)
ax_explain.text(1.5, 0.27, 'D: not dominated\n(age=1, always on front 1!)', fontsize=9, color='#2244cc')

ax_explain.set_xlabel('Age'); ax_explain.set_ylabel('Fitness (m)')
ax_explain.set_title('Dominance rules — worked examples')
ax_explain.set_xlim(0, 14); ax_explain.set_ylim(0.1, 1.0)

plt.tight_layout()
plt.show()

---
## 10. Full Evolution Loop — Annotated Pseudocode

```
INPUTS: pop=120, gen=500, sim_time=5s, workers=30, n_inject=12

── INITIALISATION ──────────────────────────────────────────
genomes[120]     ← create_connected_genome()   × 120
controllers[120] ← CPGController(n_active_tendons(g))  per g
ages[120]        ← 1
fitnesses[120]   ← evaluate_batch(genomes, controllers)  [30 workers]

── GENERATION LOOP  (repeat 500 times) ─────────────────────
for gen in 1..500:

  [A] BREED  108 children  (pop − n_inject)
      ┌─ select parent1, parent2 by tournament-3 from current pop
      ├─ child_genome = crossover_3d(p1, p2)        [70% chance]
      │                 OR copy(p1)                  [30% chance]
      ├─ child_genome = mutate(child_genome, rate=0.3)
      ├─ child_genome = keep_largest_component(child_genome)
      ├─ child_controller = crossover_controller(c1, c2, n_active)
      ├─ child_controller = mutate_controller(child_ctrl, rate=0.3)
      └─ child_age = max(parent1.age, parent2.age)

  [B] INJECT  12 random robots
      ┌─ genome     = create_connected_genome()   (20–100 voxels)
      └─ controller = CPGController(n_active)
         age        = 1

  [C] EVALUATE  120 new individuals (children + injected)
      └─ multiprocessing.Pool(30 workers).map(_eval_worker, ...)
         each worker: load MuJoCo XML → run physics → return fitness

  [D] PARETO SELECT  from pool of 240 (120 survivors + 120 new)
      ├─ Non-dominated sort on (fitness↑, age↓)
      ├─ Take front 1 entirely, then front 2, ...
      ├─ If last front overflows, keep highest-fitness from that front
      └─ Keep top 120

  [E] AGE  survivors += 1

  [F] CHECKPOINT  save best_robot_tendon.pkl  (safe Ctrl+C)

OUTPUT: best_robot_tendon.pkl  →  visualize_tendon_best.py
```

---
## 11. Parallel Evaluation Architecture

```
Main process
  │
  ├─ evaluate_batch([g0..g119], [c0..c119])
  │     │
  │     └─ multiprocessing.Pool(30 workers)
  │           │  chunksize=1  (dynamic scheduling)
  │           │
  │      Worker 0: MuJoCoTendonPhysics()  load_robot(g0) → get_fitness() → 0.0312 m
  │      Worker 1: MuJoCoTendonPhysics()  load_robot(g1) → get_fitness() → 0.0041 m
  │      Worker 2: MuJoCoTendonPhysics()  load_robot(g2) → get_fitness() → 0.0000 m
  │      ...  (30 workers in parallel)
  │      Worker 29: ...
  │
  └─ return np.array([0.0312, 0.0041, 0.0000, ...])
```

**Key design constraint:** `_eval_worker` must be a **module-level function** (not a method or lambda).
Windows uses `spawn` mode for multiprocessing (no `fork`), which requires all objects sent to
worker processes to be picklable. A class method or closure is not picklable.

**Fitness caching:** Pareto survivors from the previous generation keep their cached fitness.
Only the 120 new individuals (children + injected) need evaluation each generation.
This halves evaluation cost compared to re-evaluating everyone.

In [ ]:
# --- Parallelism speedup model ---
n_new_per_gen  = 120          # children + injected evaluated each gen
t_per_robot    = 6.5          # seconds per robot (5s sim + overhead)
overhead_frac  = 0.15         # process pool management overhead

workers = np.array([1, 2, 4, 8, 16, 30, 32])
t_seq   = n_new_per_gen * t_per_robot
t_par   = np.ceil(n_new_per_gen / workers) * t_per_robot * (1 + overhead_frac)
speedup = t_seq / t_par

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Speedup
ax1.plot(workers, speedup, 'o-', color='steelblue', lw=2, ms=8)
ax1.plot(workers, workers, '--', color='grey', lw=1, label='Ideal linear')
ax1.scatter([30], [speedup[workers==30]], s=150, color='red', zorder=5,
            label=f'Current: 30 workers = {speedup[workers==30][0]:.1f}x')
ax1.set_xlabel('Number of CPU workers')
ax1.set_ylabel('Speedup vs single worker')
ax1.set_title('Parallelism speedup (120 new robots/gen)')
ax1.legend()

# Time per generation
ax2.bar(workers.astype(str), t_par/60, color='steelblue', alpha=0.7, edgecolor='k')
ax2.axhline(t_seq/60, color='red', ls='--', lw=1.5, label=f'Sequential: {t_seq/60:.0f} min/gen')
idx30 = np.where(workers==30)[0][0]
ax2.bar([str(30)], [t_par[idx30]/60], color='#22cc44', alpha=0.9, edgecolor='k')
ax2.set_xlabel('Number of workers')
ax2.set_ylabel('Wall time per generation (min)')
ax2.set_title('Expected time per generation')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'Sequential: {t_seq:.0f}s = {t_seq/60:.1f} min/gen')
print(f'30 workers: {t_par[idx30]:.0f}s = {t_par[idx30]/60:.1f} min/gen  ({speedup[idx30]:.1f}x speedup)')
print(f'500 gen est: {500*t_par[idx30]/3600:.1f} hours')

---
## 12. Synthetic Evolution Dynamics

What to expect from a 500-generation run with Age-Fitness Pareto.
The plot below uses a synthetic model calibrated to typical soft-robot EA behaviour.

In [ ]:
np.random.seed(99)
N_GEN = 500

# Synthetic model: fitness grows in bursts (punctuated equilibria)
best_fit  = np.zeros(N_GEN)
mean_fit  = np.zeros(N_GEN)
worst_fit = np.zeros(N_GEN)
max_age_arr = np.arange(1, N_GEN+1, dtype=float)

current_best = 0.02
for g in range(N_GEN):
    if np.random.random() < 0.04 * np.exp(-g/200):
        current_best *= np.random.uniform(1.3, 2.5)
    current_best += np.random.uniform(0, 0.001)
    noise = np.random.normal(0, 0.002)
    best_fit[g]  = current_best + max(0, noise)
    mean_fit[g]  = current_best * np.random.uniform(0.25, 0.50)
    worst_fit[g] = current_best * np.random.uniform(0.01, 0.05)

def smooth(x, w=5):
    return np.convolve(x, np.ones(w)/w, mode='same')

best_fit  = smooth(best_fit,  3)
mean_fit  = smooth(mean_fit,  5)
worst_fit = smooth(worst_fit, 5)

gens = np.arange(N_GEN)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(gens, best_fit,  lw=2,   color='#22cc44', label='Best fitness')
ax1.fill_between(gens, mean_fit, best_fit, alpha=0.15, color='#22cc44')
ax1.plot(gens, mean_fit, lw=1.5, color='steelblue', label='Mean fitness', ls='--')
ax1.fill_between(gens, worst_fit, mean_fit, alpha=0.10, color='steelblue')
ax1.plot(gens, worst_fit, lw=1,  color='#cc2222', label='Worst fitness', ls=':')
ax1.set_ylabel('Fitness (m crawled in 5s)')
ax1.set_title('Synthetic evolution dynamics — 500 generations, pop=120, Age-Fitness Pareto')
ax1.legend()

for g in range(0, N_GEN, 50):
    ax1.axvline(g, color='gold', lw=0.5, alpha=0.5)

ax2.plot(gens, max_age_arr, lw=2, color='#8844cc', label='max_age (= generation number)')
ax2.fill_between(gens, 0, max_age_arr*0.4, alpha=0.15, color='#8844cc',
                 label='Typical mean age range')
ax2.set_ylabel('Age (generations)')
ax2.set_xlabel('Generation')
ax2.set_title('Age distribution — max_age grows linearly; injected robots reset diversity')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'Synthetic final best fitness: {best_fit[-1]:.3f} m')
print('NOTE: Illustrative only. Real results depend on physics, mutation luck, and time.')

---
## 13. Why Your GPU Is Not Being Used (And When It Would Be)

The entire pipeline runs on CPU. MuJoCo's standard `mj_step` is a CPU-only function.
The 30 worker processes each pin a CPU core.

### When would GPU be worth it?

| Condition | Required | Current |
|---|---|---|
| Parallel sims on GPU | ~5,000–50,000 | 120 |
| GPU utilisation at 120 | < 1% (overhead dominates) | — |
| Software stack | MuJoCo MJX (JAX) | Standard MuJoCo |
| Code changes needed | Full rewrite in JAX | — |

**MuJoCo MJX** (JAX-based GPU port) batches thousands of simulations as a single
SIMD operation on GPU. The break-even point vs 30 CPU cores is roughly **pop > 2,000**.

**A bigger grid (20×20×20) would NOT help.** Larger robots make each individual simulation
slower but don't increase parallelism. The GPU utilisation would still be <1%.

**Where your RTX A4500 WOULD help right now:**
- If you add a neural network controller (MLP, transformer) and backpropagate through it
- If you switch from voxel direct encoding to a CPPN trained with gradient descent
- CMA-ES or Adam optimisation of controller weights (GPU-accelerated matrix ops)

### Recommendation
For the current EA approach: **leave it on CPU**. 30 cores × 5s/robot gives you
~4 evaluation rounds per generation. For GPU to win, you'd need 1000+ robots,
which would require a completely different architecture (MJX + JAX).

---
## 14. Key Parameters Summary

| Category | Parameter | Value | Where to change |
|---|---|---|---|
| Grid | `VOXEL_GRID_SHAPE` | `(16,16,16)` | `genome_config.py` |
| Grid | `MIN/MAX_VOXELS_PER_ROBOT` | 20 – 100 | `genome_config.py` |
| Physics | timestep | 0.0005 s | `mujoco_tendon_converter.py` |
| Physics | voxel density | 1000 kg/m³ | `mujoco_tendon_converter.py` |
| Physics | kp active | 100 N/m | `mujoco_tendon_converter.py` |
| Physics | kp soft passive | 50 N/m | `mujoco_tendon_converter.py` |
| Physics | kv damping | 0.1 N·s/m | `mujoco_tendon_converter.py` |
| Actuation | frequency | 10 Hz | `run_tendon_evolution.py` |
| Actuation | amplitude | ±8% | `run_tendon_evolution.py` |
| EA | population size | 120 | `--pop` |
| EA | generations | 500 | `--gen` |
| EA | mutation rate | 0.3 | `DEFAULT_MUT_RATE` |
| EA | crossover rate | 0.7 | `DEFAULT_XOVER_RATE` |
| EA | injection per gen | 12 (pop//10) | `--inject` |
| EA | workers | 30 | `--workers` |
| Fitness | simulation time | 5.0 s | `--sim-time` |
| Fitness | settle time | 0.5 s | `DEFAULT_SETTLE_TIME` |

---
*Generated for run:* `python run_tendon_evolution.py --pop 120 --gen 500 --sim-time 5 --workers 30`